# Paper 3 — Persona-aware IEDI DMM

The full schema-approved demonstration persona is loaded before interpretation and the centralized DMM chooses Local, Flash or Pro from ambiguity evidence. Native-speaker validation remains external evidence.


In [ ]:
from pathlib import Path
import os
import sys

search_roots = (Path.cwd(), *Path.cwd().parents, Path("/content/iedi-mas"))
ROOT = next((path for path in search_roots if (path / "src" / "iedi").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Repository not found. Clone it and install with: pip install -e .[gemini]")
sys.path.insert(0, str(ROOT / "src"))

from iedi.codebook import Codebook
from iedi.providers import GoogleGenAIProvider, OfflineFixtureProvider
from iedi.pipeline import build_pipeline
from iedi.schemas import InterpretationRequest

codebook = Codebook.from_json(ROOT / "data" / "codebook.demo.json")
# OfflineFixtureProvider only echoes reviewed evidence; it is never empirical evidence.
# Set IEDI_LIVE_GEMINI=1 and GEMINI_API_KEY to exercise the real 2.5 Flash/Pro adapter.
LIVE_GEMINI = os.getenv("IEDI_LIVE_GEMINI") == "1"
provider = GoogleGenAIProvider() if LIVE_GEMINI else OfflineFixtureProvider()
print("provider:", "live Gemini" if LIVE_GEMINI else "offline schema fixture")

pipeline = build_pipeline("paper3", codebook=codebook, provider=provider, config_path=ROOT / "configs" / "paper3.json")


In [ ]:
ambiguous = InterpretationRequest(
    utterance="I beg",
    active_persona_ids=("ng-en-v1",),
)
result = pipeline.interpret(ambiguous)
result.to_dict()


In [ ]:
context_resolved = InterpretationRequest(
    utterance="I beg",
    active_persona_ids=("ng-en-v1",),
    supplied_tone="Casual",
    supplied_context="discourse marker used to soften commands",
)
pipeline.interpret(context_resolved).to_dict()


The paper asks for three senses but supplies two. Until a qualified annotator approves a third, the result is marked for human review rather than fabricating cultural ground truth.
